# Noticias FNSPID de las Mag7 y cobertura anual

Descarga los dos CSV públicos de **Zihan1004/FNSPID**, filtra por los símbolos de
las siete empresas del manifiesto de 10-K y mide la cobertura observada por empresa
y **año natural de publicación de la noticia**. No llama a un LLM.

**Preparación:** instalar `python -m pip install -e ".[notebook]"` y ejecutar antes
`mag7_10k_corpus.ipynb`, o copiar su `data/mag7_10k/manifest.json` al mismo lugar.
No necesita claves de API ni volver a descargar los informes SEC.

La ejecución completa descarga aproximadamente **29 GB**, además del espacio para
el subconjunto filtrado. Hugging Face reutiliza archivos completos y recupera las
descargas interrumpidas; el filtrado posterior vuelve a recorrer los CSV por bloques.
La revisión del dataset y las huellas de los archivos están fijadas para reproducibilidad.

**Alcance:** asociación por el campo `Stock_symbol`, incluidos `FB/META` y
`GOOG/GOOGL`. No se buscan nombres en el texto: noticias sin ticker o mal etiquetadas
pueden quedar fuera. El intervalo global declarado de FNSPID es 1999–2023, sin
garantía de cobertura continua por empresa. Un campo `Article` no vacío no garantiza
texto íntegro; titulares y resúmenes automáticos no se contabilizan como artículos.

Fuentes: [dataset y licencia CC BY-NC 4.0](https://huggingface.co/datasets/Zihan1004/FNSPID),
[artículo de Dong, Fan y Peng (2024)](https://arxiv.org/abs/2402.06698),
[archivos originales](https://huggingface.co/datasets/Zihan1004/FNSPID/tree/main/Stock_news).
Los datos se guardan bajo `data/`, excluido de Git; no se incorporan artículos al notebook.

In [1]:
from __future__ import annotations

import gzip
import hashlib
import json
import re
import sqlite3
from contextlib import closing
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urldefrag

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download
from IPython.display import display

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src/data_extraction").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Abre el notebook desde el proyecto o desde notebooks/.")

MANIFEST_PATH = PROJECT_ROOT / "data/mag7_10k/manifest.json"
OUTPUT_DIR = PROJECT_ROOT / "data/mag7_fnspid"
RAW_DIR = OUTPUT_DIR / "raw"
CHUNK_SIZE = 20_000
REPO_ID = "Zihan1004/FNSPID"
REVISION = "bf9189c41527198897d1af3e17b1a0095279fc45"
SOURCE_FILES = {
    "Stock_news/nasdaq_exteral_data.csv": {
        "size": 23232979597,
        "sha256": "1a7a3eb8e6b97ec19f286f2cfca3371542bddb272ab1eb8f36e33ad98fa5c4da",
    },
    "Stock_news/All_external.csv": {
        "size": 5731397037,
        "sha256": "5d4c018036bd82ca821da71b7a9c0c7db3289642e0fc6f897ea69f4a0c5135c3",
    },
}
FNSPID_FIRST_YEAR, FNSPID_LAST_YEAR = 1999, 2023
ALIASES = {
    "AAPL": "AAPL", "MSFT": "MSFT", "AMZN": "AMZN", "NVDA": "NVDA",
    "TSLA": "TSLA", "META": "META", "FB": "META", "GOOG": "GOOGL", "GOOGL": "GOOGL",
}

def sha256_file(path):
    """Calcula la huella sin cargar el archivo en memoria."""
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

## 1. Empresas y años derivados del manifiesto

La ventana de cada empresa abarca años naturales completos: comienza en el año de
su primer cierre fiscal disponible (o presentación, si falta el cierre) y termina
en el año de su última presentación. Así se incluye el ejercicio del primer 10-K.
No se confunden estos años con los ejercicios fiscales y no se recortan las noticias
al día exacto de presentación. El último año puede ser parcial en la fuente.

Se usa `filings`, el inventario esperado del manifiesto, y se informa por separado
del estado de las descargas SEC. Un error de inventario impide definir una cobertura
completa; un error de descarga no elimina silenciosamente años de la ventana.

In [2]:
def load_periods(manifest_path):
    """Obtiene ventanas y estados del inventario SEC existente."""
    if not manifest_path.is_file():
        raise FileNotFoundError("Falta data/mag7_10k/manifest.json; ejecuta primero el notebook de 10-K.")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("inventory_errors"):
        raise ValueError("El manifiesto tiene errores de inventario SEC; resuélvelos antes de medir cobertura.")
    filings = pd.DataFrame(manifest["filings"])
    required = {"study_ticker", "study_company", "cik", "form", "accession_number", "filing_date", "report_date"}
    if not required.issubset(filings.columns):
        raise ValueError(f"Faltan columnas del manifiesto: {sorted(required - set(filings.columns))}")
    filings = filings.loc[filings["form"].isin(["10-K", "10-K405"])].copy()
    filings = filings.drop_duplicates(["study_ticker", "accession_number"])
    if set(filings["study_ticker"]) != set(ALIASES.values()):
        raise ValueError("El inventario debe contener exactamente las siete empresas del corpus Mag7.")
    for field in ["filing_date", "report_date"]:
        filings[field] = pd.to_datetime(filings[field], utc=True, errors="coerce")
    if filings["filing_date"].isna().any():
        raise ValueError("Hay fechas de presentación inválidas en el manifiesto.")
    filings["window_start"] = filings[["report_date", "filing_date"]].min(axis=1)
    periods = filings.groupby("study_ticker").agg(
        empresa=("study_company", "first"), informes=("accession_number", "size"),
        ciks=("cik", lambda s: ", ".join(sorted(set(s.astype(str))))),
        inicio=("window_start", "min"), primera_presentacion=("filing_date", "min"),
        ultima_presentacion=("filing_date", "max"),
    )
    periods["primer_ano"] = periods["inicio"].dt.year
    periods["ultimo_ano"] = periods["ultima_presentacion"].dt.year
    downloads = pd.DataFrame(manifest.get("downloads", []))
    return manifest, filings, periods, downloads

manifest, filings, periods, downloads = load_periods(MANIFEST_PATH)
print("Instantánea SEC:", manifest.get("queried_at_utc", "no indicada"))
print("Informes del inventario:", len(filings))
display(periods)
if not downloads.empty:
    display(pd.crosstab(downloads["study_ticker"], downloads["download_status"]))

Instantánea SEC: 2026-10-11T15:30:10.552211+00:00
Informes del inventario: 174


,empresa,informes,ciks,inicio,primera_presentacion,ultima_presentacion,primer_ano,ultimo_ano
study_ticker,,,,,,,,
AAPL,Apple,32,0000320193,1994-09-30 00:00:00+00:00,1994-12-13 00:00:00+00:00,2025-10-31 00:00:00+00:00,1994,2025
AMZN,Amazon,29,0001018724,1997-12-31 00:00:00+00:00,1998-03-30 00:00:00+00:00,2026-02-06 00:00:00+00:00,1997,2026
GOOGL,Alphabet / Google,22,"0001288776, 0001652044",2004-12-31 00:00:00+00:00,2005-03-30 00:00:00+00:00,2026-02-05 00:00:00+00:00,2004,2026
META,Meta / Facebook,14,0001326801,2012-12-31 00:00:00+00:00,2013-02-01 00:00:00+00:00,2026-01-29 00:00:00+00:00,2012,2026
MSFT,Microsoft,33,0000789019,1994-06-30 00:00:00+00:00,1994-09-27 00:00:00+00:00,2026-07-29 00:00:00+00:00,1994,2026
NVDA,NVIDIA,28,0001045810,1999-01-31 00:00:00+00:00,1999-04-29 00:00:00+00:00,2026-02-25 00:00:00+00:00,1999,2026
TSLA,Tesla,16,0001318605,2010-12-31 00:00:00+00:00,2011-03-03 00:00:00+00:00,2026-01-29 00:00:00+00:00,2010,2026


download_status,cached
study_ticker,
AAPL,32
AMZN,29
GOOGL,22
META,14
MSFT,33
NVDA,28
TSLA,16


## 2. Descarga pública y comprobación de integridad

Esta celda inicia la descarga completa. Reejecutarla reutiliza la caché local.
Se comprueba tamaño y SHA-256 antes de procesar cada CSV. La lectura para verificar
las huellas también tarda, aunque el archivo ya esté descargado.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
source_paths = {}
for filename, expected in SOURCE_FILES.items():
    print(f"Descarga/caché: {filename} ({expected['size'] / 1e9:.2f} GB)")
    path = Path(hf_hub_download(
        repo_id=REPO_ID, repo_type="dataset", filename=filename,
        revision=REVISION, local_dir=RAW_DIR, token=False,
    ))
    if path.stat().st_size != expected["size"] or sha256_file(path) != expected["sha256"]:
        raise ValueError(f"Archivo no íntegro: {filename}. No se calculará cobertura con él.")
    source_paths[filename] = path
    print("Integridad comprobada.")

## 3. Filtrado por bloques y deduplicación en disco

Se conserva el texto original de las coincidencias en SQLite, sin cargar todos los
artículos en RAM. Los símbolos múltiples se separan; una noticia puede pertenecer
a varias empresas, pero `GOOG/GOOGL` solo genera una asociación a Alphabet.

La clave de deduplicación es **empresa + URL** (sin fragmento `#`). Sin URL se usa
empresa + titular normalizado + fecha/hora. No se eliminan parámetros de consulta
ni se realiza deduplicación semántica entre medios. Entre duplicados se conserva la
fila con el `Article` más largo; en empate, la primera. Su fecha es la de esa fila.
`provenance` registra todas las coincidencias y sus fechas para auditar discrepancias.

Las fechas inválidas y las filas sin URL ni titular se contabilizan y excluyen. Los
errores de lectura CSV detienen el proceso: no se saltan silenciosamente. Se excluyen
noticias fuera de la ventana SEC de cada empresa y se cuenta cuántas son.

La base anterior solo se sustituye al terminar ambos archivos. Una interrupción
requiere repetir el filtrado, pero no la descarga. La cobertura solo se calcula
después de una lectura completa; no hay modo muestra que produzca cifras finales.

In [ ]:
FIELDS = ["Date", "Article_title", "Stock_symbol", "Url", "Publisher", "Article"]
EMPTY_VALUES = {"", "nan", "none", "null", "n/a"}

def filter_news(source_paths, periods, output_dir, chunk_size=CHUNK_SIZE):
    """Filtra ambos archivos y conserva deduplicación y procedencia en SQLite."""
    output_dir.mkdir(parents=True, exist_ok=True)
    database = output_dir / "news.sqlite"
    staging = output_dir / "news.sqlite.part"
    staging.unlink(missing_ok=True)
    stats = []
    bounds = periods[["primer_ano", "ultimo_ano"]].to_dict("index")
    with closing(sqlite3.connect(staging)) as conn:
        conn.executescript("""
            CREATE TABLE news (
                study_ticker TEXT, article_key TEXT, published_utc TEXT, year INTEGER,
                title TEXT, url TEXT, publisher TEXT, article TEXT,
                source_file TEXT, source_row INTEGER,
                PRIMARY KEY (study_ticker, article_key)
            );
            CREATE TABLE provenance (
                study_ticker TEXT, article_key TEXT, source_file TEXT, source_row INTEGER,
                original_symbol TEXT, published_utc TEXT, article_chars INTEGER
            );
        """)
        upsert = """
            INSERT INTO news VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            ON CONFLICT(study_ticker, article_key) DO UPDATE SET
                published_utc=excluded.published_utc, year=excluded.year,
                title=excluded.title, url=excluded.url, publisher=excluded.publisher,
                article=excluded.article, source_file=excluded.source_file,
                source_row=excluded.source_row
            WHERE length(excluded.article) > length(news.article)
        """
        for filename, path in source_paths.items():
            counts = dict(source_file=filename, filas_leidas=0, filas_sin_ticker=0,
                          filas_mag7=0, asociaciones_fecha_invalida=0,
                          asociaciones_fuera_ventana=0, asociaciones_sin_identificador=0,
                          asociaciones_conservadas=0)
            for block_number, chunk in enumerate(pd.read_csv(
                path, usecols=FIELDS, dtype=str, keep_default_na=False,
                chunksize=chunk_size, on_bad_lines="error", encoding="utf-8",
            ), start=1):
                chunk["source_row"] = range(counts["filas_leidas"] + 1,
                                             counts["filas_leidas"] + len(chunk) + 1)
                counts["filas_leidas"] += len(chunk)
                for field in FIELDS:
                    chunk[field] = chunk[field].str.strip()
                    chunk.loc[chunk[field].str.lower().isin(EMPTY_VALUES), field] = ""
                counts["filas_sin_ticker"] += int(chunk["Stock_symbol"].eq("").sum())
                chunk["study_ticker"] = chunk["Stock_symbol"].map(
                    lambda s: sorted({ALIASES[t] for t in re.findall(r"[A-Z][A-Z0-9.-]*", s.upper())
                                      if t in ALIASES})
                )
                selected = chunk.loc[chunk["study_ticker"].map(bool)].copy()
                counts["filas_mag7"] += len(selected)
                selected = selected.explode("study_ticker", ignore_index=True)
                dates = pd.to_datetime(selected["Date"], format="%Y-%m-%d %H:%M:%S UTC",
                                       errors="coerce", utc=True)
                # Acepta también fechas ISO sin inferir un único formato para todo el bloque.
                missing = dates.isna()
                if missing.any():
                    dates.loc[missing] = pd.to_datetime(
                        selected.loc[missing, "Date"].map(
                            lambda s: pd.to_datetime(s, errors="coerce", utc=True)
                        ), utc=True,
                    )
                selected["parsed_date"] = dates
                counts["asociaciones_fecha_invalida"] += int(dates.isna().sum())
                rows, references = [], []
                for item in selected.loc[dates.notna()].to_dict("records"):
                    ticker, date = item["study_ticker"], item["parsed_date"]
                    if not bounds[ticker]["primer_ano"] <= date.year <= bounds[ticker]["ultimo_ano"]:
                        counts["asociaciones_fuera_ventana"] += 1
                        continue
                    url, title = urldefrag(item["Url"])[0], item["Article_title"]
                    if not url and not title:
                        counts["asociaciones_sin_identificador"] += 1
                        continue
                    published = date.isoformat()
                    identity = ("url:" + url) if url else (
                        "title:" + " ".join(title.casefold().split()) + "|" + published
                    )
                    key = hashlib.sha256(identity.encode("utf-8")).hexdigest()
                    rows.append((ticker, key, published, date.year, title, url,
                                 item["Publisher"], item["Article"], filename, item["source_row"]))
                    references.append((ticker, key, filename, item["source_row"],
                                       item["Stock_symbol"], published, len(item["Article"])))
                conn.executemany(upsert, rows)
                conn.executemany("INSERT INTO provenance VALUES (?, ?, ?, ?, ?, ?, ?)", references)
                conn.commit()
                counts["asociaciones_conservadas"] += len(rows)
                if block_number % 10 == 0:
                    print(f"{filename}: {counts['filas_leidas']:,} filas leídas")
            stats.append(counts)
            print(f"Completado {filename}: {counts['filas_leidas']:,} filas")
        conn.execute("CREATE INDEX provenance_article ON provenance(study_ticker, article_key)")
        conn.commit()
    staging.replace(database)
    return database, pd.DataFrame(stats)

# Ejecución completa: no convertir una descarga parcial en cobertura final.
if set(source_paths) != set(SOURCE_FILES):
    raise RuntimeError("Completa y verifica ambos CSV antes de calcular la cobertura.")
database, scan_stats = filter_news(source_paths, periods, OUTPUT_DIR)
display(scan_stats)

## 4. Cobertura, huecos y procedencia

Cada fila corresponde a una empresa y un año de su ventana. `noticias_unicas` es el
número de asociaciones deduplicadas según la regla anterior; sumar empresas puede
contar una misma noticia varias veces. `con_articulo` mide texto no vacío, no calidad
ni integridad. Sin noticias, el porcentaje con artículo queda indefinido.

`sin_noticias_recuperadas` no significa ausencia de acontecimientos ni ausencia de
cobertura en otros medios. Los años fuera de 1999–2023 se identifican por separado;
si hubiera registros observados fuera del intervalo declarado, también se muestran.
Se informa de URL repetidas con fechas distintas antes de interpretar las series.

In [ ]:
def build_coverage(database, periods, filings):
    """Incluye ceros dentro de cada ventana, sin inventar cobertura fuera de ella."""
    with closing(sqlite3.connect(database)) as conn:
        observed = pd.read_sql_query("""
            SELECT study_ticker, year AS ano, COUNT(*) AS noticias_unicas,
                   SUM(length(article) > 0) AS con_articulo,
                   COUNT(DISTINCT NULLIF(publisher, '')) AS editores_identificados,
                   COUNT(DISTINCT substr(published_utc, 1, 7)) AS meses_con_noticias,
                   MIN(published_utc) AS primera_noticia, MAX(published_utc) AS ultima_noticia
            FROM news GROUP BY study_ticker, year
        """, conn)
        conflicts = pd.read_sql_query("""
            SELECT study_ticker, article_key, COUNT(*) AS coincidencias,
                   COUNT(DISTINCT substr(published_utc, 1, 10)) AS fechas_distintas
            FROM provenance GROUP BY study_ticker, article_key
            HAVING COUNT(DISTINCT substr(published_utc, 1, 10)) > 1
        """, conn)
    grid = pd.DataFrame([
        {"study_ticker": ticker, "ano": year}
        for ticker, p in periods.iterrows()
        for year in range(int(p["primer_ano"]), int(p["ultimo_ano"]) + 1)
    ])
    coverage = grid.merge(observed, how="left", on=["study_ticker", "ano"])
    count_fields = ["noticias_unicas", "con_articulo", "editores_identificados", "meses_con_noticias"]
    coverage[count_fields] = coverage[count_fields].fillna(0).astype(int)
    annual_filings = (filings.assign(ano=filings["filing_date"].dt.year)
                      .groupby(["study_ticker", "ano"]).size().rename("informes_presentados"))
    coverage = coverage.join(annual_filings, on=["study_ticker", "ano"])
    coverage["informes_presentados"] = coverage["informes_presentados"].fillna(0).astype(int)
    coverage["pct_con_articulo"] = (
        100 * coverage["con_articulo"] / coverage["noticias_unicas"].replace(0, np.nan)
    )
    declared = coverage["ano"].between(FNSPID_FIRST_YEAR, FNSPID_LAST_YEAR)
    coverage["estado"] = np.select(
        [coverage["noticias_unicas"].gt(0) & declared,
         coverage["noticias_unicas"].gt(0) & ~declared, ~declared],
        ["con_noticias", "observada_fuera_periodo_declarado", "fuera_periodo_declarado_fnspid"],
        default="sin_noticias_recuperadas",
    )
    return coverage, conflicts

coverage, date_conflicts = build_coverage(database, periods, filings)
summary = coverage.groupby("study_ticker").agg(
    noticias_unicas=("noticias_unicas", "sum"), con_articulo=("con_articulo", "sum"),
    anos_en_ventana=("ano", "size"), anos_con_noticias=("noticias_unicas", lambda s: int(s.gt(0).sum())),
)
summary["pct_con_articulo"] = 100 * summary["con_articulo"] / summary["noticias_unicas"].replace(0, np.nan)
display(summary.round(2))
display(coverage)
print("Artículos asociados con discrepancias de fecha:", len(date_conflicts))
display(date_conflicts.head(20))

In [ ]:
matrix = coverage.pivot(index="study_ticker", columns="ano", values="noticias_unicas")
fig, ax = plt.subplots(figsize=(16, 4))
cmap = plt.get_cmap("YlGnBu").copy()
cmap.set_bad("#dddddd")
heatmap = ax.imshow(np.log1p(matrix.to_numpy(dtype=float)), aspect="auto", cmap=cmap)
ax.set_xticks(range(len(matrix.columns)), matrix.columns, rotation=90)
ax.set_yticks(range(len(matrix.index)), matrix.index)
ax.set_xlabel("Año natural de publicación de la noticia")
ax.set_title("Noticias FNSPID por empresa y año · gris: fuera de la ventana SEC")
fig.colorbar(heatmap, ax=ax, label="log(1 + noticias únicas)")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "cobertura.png", dpi=160)
plt.show()

display(coverage.loc[coverage["noticias_unicas"].eq(0),
                     ["study_ticker", "ano", "informes_presentados", "estado"]])

## 5. Guardar resultados reproducibles

- `news.sqlite`: artículos deduplicados (`news`) y todas sus coincidencias (`provenance`).
- `mag7_news.jsonl.gz`: exportación comprimida del subconjunto, un registro por empresa y noticia.
- `cobertura_empresa_ano.csv`, `resumen_empresa.csv`, `huecos.csv`, `conflictos_fecha.csv` y `cobertura.png`.
- `periodos_sec.csv`, `diagnostico_fuentes.csv` y `run_manifest.json`: ventanas, descartes, fuentes y huellas.

El manifiesto identifica la revisión FNSPID y la huella del manifiesto SEC. Los
recuentos describen este dataset y este filtro, no todas las noticias publicadas
sobre las Mag7. Revisar las discrepancias de fecha antes de enlazar acontecimientos
con un 10-K; una noticia posterior no debe entrar como evidencia disponible antes
de su publicación.

In [ ]:
coverage.to_csv(OUTPUT_DIR / "cobertura_empresa_ano.csv", index=False)
summary.to_csv(OUTPUT_DIR / "resumen_empresa.csv")
coverage.loc[coverage["noticias_unicas"].eq(0)].to_csv(OUTPUT_DIR / "huecos.csv", index=False)
date_conflicts.to_csv(OUTPUT_DIR / "conflictos_fecha.csv", index=False)
periods.to_csv(OUTPUT_DIR / "periodos_sec.csv")
scan_stats.to_csv(OUTPUT_DIR / "diagnostico_fuentes.csv", index=False)

export_path = OUTPUT_DIR / "mag7_news.jsonl.gz"
with closing(sqlite3.connect(database)) as conn, gzip.open(export_path.with_suffix(".gz.part"), "wt", encoding="utf-8") as stream:
    conn.row_factory = sqlite3.Row
    for row in conn.execute("SELECT * FROM news ORDER BY study_ticker, year, published_utc, article_key"):
        stream.write(json.dumps(dict(row), ensure_ascii=False) + "\n")
export_path.with_suffix(".gz.part").replace(export_path)

run_manifest = {
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "scan_complete": True,
    "sec_manifest": MANIFEST_PATH.relative_to(PROJECT_ROOT).as_posix(),
    "sec_manifest_sha256": sha256_file(MANIFEST_PATH),
    "sec_queried_at_utc": manifest.get("queried_at_utc"),
    "dataset": REPO_ID, "revision": REVISION, "source_files": SOURCE_FILES,
    "license_reference": f"https://huggingface.co/datasets/{REPO_ID}/blob/{REVISION}/README.md",
    "aliases": ALIASES, "chunk_size": CHUNK_SIZE,
    "window": "Años naturales completos desde primer cierre/presentación hasta última presentación SEC",
    "deduplication": "Empresa + URL sin fragmento; sin URL, titular normalizado + fecha/hora; se conserva mayor Article",
    "news_company_associations": int(coverage["noticias_unicas"].sum()),
    "date_conflicts": len(date_conflicts),
    "source_diagnostics": scan_stats.to_dict("records"),
    "notebook_sha256": sha256_file(PROJECT_ROOT / "notebooks/mag7_fnspid_news.ipynb"),
}
(OUTPUT_DIR / "run_manifest.json").write_text(
    json.dumps(run_manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("Resultados guardados en", OUTPUT_DIR.relative_to(PROJECT_ROOT).as_posix())